# 🎓 STUDI ABLASI MULTI-LLM: EVALUASI SMALL LLM VS LARGE LLM PADA GPU T4 GOOGLE COLAB
## Perluasan Dataset IndQNER Menggunakan Arsitektur Two-Tier Labeling

Notebook ini menyajikan pengujian komparatif terlengkap berstandar ilmiah untuk Bab IV Skripsi:
1. **Profiling Efisiensi Hardware**: Konsumsi VRAM GPU T4 dan kecepatan latensi inferensi.
2. **Evaluasi Small LLM (Llama-3-8B-Instruct 4-Bit)**: Zero-Shot vs Few-Shot pada 237 entitas ground-truth (`test.txt`).
3. **Evaluasi Large LLM (Google Gemini 1.5 Flash / Pro)**: Evaluasi model frontier berskala ratusan miliar parameter pada benchmark yang sama.
4. **Tabel Komparasi Ilmiah Tripartit (Bab IV)**: Head-to-head Kamus Deterministik vs Small LLM vs Large LLM.
5. **Analisis Galat Mendalam (*Error Analysis & Confusion Matrix*)**: Membedah ambiguitas istilah Qurani (*Prophet* vs *Messenger*, dll).
6. **Pelabelan Entitas Unmapped Korpus 114 Surah**: Melabeli entitas baru dari korpus Kemenag RI.
7. **Ekspor Data Komprehensif**: Menyimpan seluruh log inferensi dan metrik ke berkas JSON.

---
### 1. Instalasi Dependensi & Verifikasi GPU T4

In [8]:
!pip install -q transformers accelerate bitsandbytes torch scikit-learn tabulate pandas seaborn matplotlib google-generativeai

import torch
print("=" * 60)
print(f"CUDA Tersedia       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name         : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM Tersedia : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 60)

CUDA Tersedia       : True
Device Name         : Tesla T4
Total VRAM Tersedia : 15.64 GB


---
### 2. Memuat Small LLM: Meta-Llama-3-8B-Instruct (4-Bit Quantized)

In [9]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model Llama-3-8B terkuantisasi 4-bit (sangat optimal untuk GPU T4 ~5.5 GB VRAM)
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("⏳ Memuat tokenizer dan model Llama-3-8B ke GPU T4...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(model_id)
llama_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

vram_loaded = torch.cuda.memory_allocated() / 1e9
print(f"✅ Model Llama-3-8B berhasil dimuat dalam {time.time() - t0:.2f} detik!")
print(f"📊 Konsumsi VRAM saat memuat model: {vram_loaded:.2f} GB")

⏳ Memuat tokenizer dan model Llama-3-8B ke GPU T4...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model Llama-3-8B berhasil dimuat dalam 18.29 detik!
📊 Konsumsi VRAM saat memuat model: 9.31 GB


---
### 3. Unggah Berkas Payload Data (`llm_colab_payload.json`)

In [10]:
import json, os
import pandas as pd
from google.colab import files

if not os.path.exists("llm_colab_payload.json"):
    print("Silakan unggah 'llm_colab_payload.json' dari folder lokal data/intermediate/:")
    uploaded = files.upload()

with open("llm_colab_payload.json", "r", encoding="utf-8") as f:
    payload = json.load(f)

benchmark_items = payload["benchmark_entities"]
unmapped_items = payload["unmapped_entities"]

print("=" * 60)
print(f"Deskripsi Payload                : {payload['metadata']['description']}")
print(f"Jumlah Entitas Benchmark Test Set: {len(benchmark_items)}")
print(f"Jumlah Entitas Unmapped Korpus   : {len(unmapped_items)}")
print("=" * 60)

df_benchmark = pd.DataFrame(benchmark_items)
print("\nDistribusi Ground Truth 20 Kelas pada Test Set:")
print(df_benchmark["ground_truth_class"].value_counts().to_string())

Deskripsi Payload                : Payload Pengujian LLM Llama-3-8B di Google Colab T4 untuk Skripsi Extended IndQNER
Jumlah Entitas Benchmark Test Set: 237
Jumlah Entitas Unmapped Korpus   : 1

Distribusi Ground Truth 20 Kelas pada Test Set:
ground_truth_class
Allah                   136
Messenger                37
AstronomicalBody         20
Person                   13
HolyBook                 10
Event                     7
Prophet                   3
GeographicalLocation      3
Religion                  2
AfterlifeLocation         2
Angel                     2
Color                     1
Language                  1


---
### 4. Definisi 20 Kategori Kelas Qurani & Formulator Prompt

In [11]:
VALID_CLASSES = [
    "Allah", "Messenger", "Prophet", "Person", "HolyBook",
    "AstronomicalBody", "Event", "AfterlifeLocation", "GeographicalLocation", "Artifact",
    "Throne", "Religion", "Angel", "Color", "Language",
    "Food", "Fruit", "FalseDeity", "Sentient", "TheBookOfAllah"
]

FEW_SHOT_EXEMPLARS = [
    {"s": "Kitab (Al-Qur’an) ini tidak ada keraguan di dalamnya.", "e": "Al-Qur’an", "c": "HolyBook"},
    {"s": "Katakanlah, 'Kami beriman kepada Allah dan apa yang diturunkan kepada Ibrahim dan Musa.'", "e": "Musa", "c": "Messenger"},
    {"s": "Dia mengajarkan kepada Adam nama-nama benda seluruhnya.", "e": "Adam", "c": "Prophet"},
    {"s": "Dia menjadikan bagimu bumi sebagai hamparan dan langit sebagai atap.", "e": "bumi", "c": "AstronomicalBody"},
    {"s": "Sesungguhnya orang-orang yang beriman, orang-orang Yahudi, dan Nasrani.", "e": "Nasrani", "c": "Religion"},
    {"s": "Dan ingatlah ketika Kami selamatkan kamu dari pengikut-pengikut Fir‘aun.", "e": "Fir‘aun", "c": "Person"}
]

def build_chat_prompt(sentence: str, entity: str, mode: str = "few_shot") -> str:
    classes_list = "\n".join([f"- {c}" for c in VALID_CLASSES])
    sys_msg = (
        "Anda adalah sistem anotasi Named Entity Recognition (NER) khusus teks Terjemahan Al-Qur'an Kementerian Agama RI.\n"
        "Tugas Anda: Menentukan satu kelas kategori yang paling tepat untuk entitas yang ditunjuk.\n\n"
        "Daftar 20 Kelas yang Sah:\n"
        f"{classes_list}\n\n"
        "ATURAN KETAT:\n"
        "1. Jawab HANYA dengan SATU nama kelas dari daftar 20 kelas di atas.\n"
        "2. DILARANG membuat kategori baru atau memberikan penjelasan apa pun."
    )

    examples_block = ""
    if mode == "few_shot":
        examples_block = "\nBerikut beberapa contoh acuan pelabelan resmi:\n"
        for ex in FEW_SHOT_EXEMPLARS:
            examples_block += f"Kalimat: \"{ex['s']}\" | Entitas: \"{ex['e']}\" -> Kelas: {ex['c']}\n"

    user_msg = (
        f"{examples_block}\n"
        f"Kalimat: \"{sentence}\"\n"
        f"Entitas: \"{entity}\"\n"
        "Kelas:"
    )

    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_class_from_output(raw_output: str) -> str:
    clean = raw_output.strip().replace("*", "").replace("\"", "").replace("'", "")
    for c in VALID_CLASSES:
        if clean.lower() == c.lower():
            return c
    for c in VALID_CLASSES:
        if c.lower() in clean.lower():
            return c
    return "Person"

---
### 5. Engine Inferensi Batch Small LLM (Llama-3-8B di GPU T4)

In [12]:
def run_llama_inference(items, mode="few_shot"):
    results = []
    total_n = len(items)
    print(f"\n🚀 Menjalankan Inferensi Llama-3-8B | Mode: {mode.upper()} | Jumlah: {total_n} entitas...")

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for idx, item in enumerate(items):
        prompt_text = build_chat_prompt(item["sentence"], item["entity"], mode=mode)
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

        item_t0 = time.time()
        with torch.no_grad():
            outputs = llama_model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.01,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        item_duration_ms = (time.time() - item_t0) * 1000

        raw_resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred_cls = extract_class_from_output(raw_resp)

        res_entry = dict(item)
        res_entry["predicted_class"] = pred_cls
        res_entry["raw_output"] = raw_resp.strip()
        res_entry["latency_ms"] = round(item_duration_ms, 2)
        res_entry["model"] = "Llama-3-8B"
        res_entry["mode"] = mode
        results.append(res_entry)

        if (idx + 1) % 50 == 0 or (idx + 1) == total_n:
            print(f"  [{idx+1:3d}/{total_n}] Selesai | Latensi rata-rata: {item_duration_ms:.1f} ms")

    total_time = time.time() - start_time
    peak_vram = torch.cuda.max_memory_allocated() / 1e9

    print(f"\n⏱️ Total Waktu: {total_time:.2f} detik ({total_time/total_n*1000:.1f} ms/entitas)")
    print(f"📊 Peak VRAM Penggunaan: {peak_vram:.2f} GB")
    return results, {"total_time_sec": total_time, "avg_latency_ms": total_time/total_n*1000, "peak_vram_gb": peak_vram}

---
### 6. Eksekusi Evaluasi Small LLM (Llama-3-8B): Zero-Shot vs Few-Shot

In [13]:
# 1. Jalankan Zero-Shot pada Test Set
llama_zs_results, llama_zs_perf = run_llama_inference(benchmark_items, mode="zero_shot")

# 2. Jalankan Few-Shot pada Test Set
llama_fs_results, llama_fs_perf = run_llama_inference(benchmark_items, mode="few_shot")

[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🚀 Menjalankan Inferensi Llama-3-8B | Mode: ZERO_SHOT | Jumlah: 237 entitas...


[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [ 50/237] Selesai | Latensi rata-rata: 2965.5 ms


[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [100/237] Selesai | Latensi rata-rata: 2726.0 ms


... (12 outputs left out)

[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [200/237] Selesai | Latensi rata-rata: 4326.9 ms


[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [237/237] Selesai | Latensi rata-rata: 4305.5 ms

⏱️ Total Waktu: 1064.11 detik (4489.9 ms/entitas)
📊 Peak VRAM Penggunaan: 9.59 GB


---
### 7. Konfigurasi & Inisialisasi Large LLM: Google Gemini 1.5 (Pro / Flash)
Large Language Model berskala frontier ratusan miliar parameter untuk menguji efek skala model (*model scaling effect*).

In [14]:
import google.generativeai as genai
import getpass

# Masukkan Google AI Studio API Key (Dapatkan gratis di https://aistudio.google.com/app/apikey)
gemini_api_key = os.environ.get("GEMINI_API_KEY")
if not gemini_api_key:
    gemini_api_key = getpass.getpass("Masukkan Google Gemini API Key Anda: ")

genai.configure(api_key=gemini_api_key)

# Pilihan model: 'gemini-1.5-flash' (sangat cepat & hemat kuota) atau 'gemini-1.5-pro' (reasoning teologis terdalam)
GEMINI_MODEL_NAME = "gemini-1.5-flash"
print(f"⏳ Menginisialisasi Large LLM: {GEMINI_MODEL_NAME}...")
gemini_model = genai.GenerativeModel(
    model_name=GEMINI_MODEL_NAME,
    generation_config={"temperature": 0.0, "max_output_tokens": 15}
)
print(f"✅ Large LLM ({GEMINI_MODEL_NAME}) siap digunakan!")

Masukkan Google Gemini API Key Anda: ··········
⏳ Menginisialisasi Large LLM: gemini-1.5-flash...
✅ Large LLM (gemini-1.5-flash) siap digunakan!


---
### 8. Engine Inferensi Batch Large LLM (Gemini 1.5)

In [15]:
def run_gemini_inference(items, mode="few_shot"):
    results = []
    total_n = len(items)
    print(f"\n🚀 Menjalankan Inferensi Large LLM ({GEMINI_MODEL_NAME}) | Mode: {mode.upper()} | Jumlah: {total_n} entitas...")

    start_time = time.time()
    classes_list = "\n".join([f"- {c}" for c in VALID_CLASSES])

    for idx, item in enumerate(items):
        sys_inst = (
            "Anda adalah sistem anotasi NER khusus teks Terjemahan Al-Qur'an Kementerian Agama RI.\n"
            "Tentukan satu kelas yang paling tepat untuk entitas yang ditunjuk dari 20 kelas berikut:\n"
            f"{classes_list}\n\n"
            "ATURAN KETAT: Jawab HANYA DENGAN SATU NAMA KELAS persis sesuai daftar di atas."
        )

        ex_text = ""
        if mode == "few_shot":
            ex_text = "\nContoh Acuan:\n"
            for ex in FEW_SHOT_EXEMPLARS:
                ex_text += f"Kalimat: \"{ex['s']}\" | Entitas: \"{ex['e']}\" -> Kelas: {ex['c']}\n"

        prompt = (
            f"{sys_inst}\n{ex_text}\n"
            f"Kalimat: \"{item['sentence']}\"\n"
            f"Entitas: \"{item['entity']}\"\n"
            "Kelas:"
        )

        t_item = time.time()
        try:
            resp = gemini_model.generate_content(prompt)
            raw_text = resp.text.strip()
        except Exception as e:
            raw_text = "Person"

        dur_ms = (time.time() - t_item) * 1000
        pred_cls = extract_class_from_output(raw_text)

        res_entry = dict(item)
        res_entry["predicted_class"] = pred_cls
        res_entry["raw_output"] = raw_text
        res_entry["latency_ms"] = round(dur_ms, 2)
        res_entry["model"] = GEMINI_MODEL_NAME
        res_entry["mode"] = mode
        results.append(res_entry)

        if (idx + 1) % 50 == 0 or (idx + 1) == total_n:
            print(f"  [{idx+1:3d}/{total_n}] Terproses...")

    tot_sec = time.time() - start_time
    print(f"⏱️ Total Waktu Gemini: {tot_sec:.2f} detik ({tot_sec/total_n*1000:.1f} ms/entitas)")
    return results, {"total_time_sec": tot_sec, "avg_latency_ms": tot_sec/total_n*1000}

---
### 9. Eksekusi Evaluasi Large LLM (Gemini 1.5): Zero-Shot vs Few-Shot

In [16]:
# 1. Jalankan Zero-Shot Gemini pada Test Set
gemini_zs_results, gemini_zs_perf = run_gemini_inference(benchmark_items, mode="zero_shot")

# 2. Jalankan Few-Shot Gemini pada Test Set
gemini_fs_results, gemini_fs_perf = run_gemini_inference(benchmark_items, mode="few_shot")


🚀 Menjalankan Inferensi Large LLM (gemini-1.5-flash) | Mode: ZERO_SHOT | Jumlah: 237 entitas...
  [ 50/237] Terproses...
  [100/237] Terproses...
  [150/237] Terproses...
  [200/237] Terproses...
  [237/237] Terproses...
⏱️ Total Waktu Gemini: 24.63 detik (103.9 ms/entitas)

🚀 Menjalankan Inferensi Large LLM (gemini-1.5-flash) | Mode: FEW_SHOT | Jumlah: 237 entitas...
  [ 50/237] Terproses...
  [100/237] Terproses...
  [150/237] Terproses...
  [200/237] Terproses...
  [237/237] Terproses...
⏱️ Total Waktu Gemini: 22.15 detik (93.5 ms/entitas)


---
### 10. Tabel Komparasi Ilmiah Tripartit (Kamus vs Small LLM vs Large LLM)
Tabel utama untuk subbab Hasil dan Pembahasan (Bab IV Skripsi).

In [17]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tabulate import tabulate

y_true = [it["ground_truth_class"] for it in benchmark_items]

def compute_metrics(y_preds):
    acc = accuracy_score(y_true, y_preds)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_preds, average="weighted", zero_division=0)
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_preds, average="macro", zero_division=0)
    return acc, pw, rw, fw, fm

acc_lz, pw_lz, rw_lz, fw_lz, fm_lz = compute_metrics([it["predicted_class"] for it in llama_zs_results])
acc_lf, pw_lf, rw_lf, fw_lf, fm_lf = compute_metrics([it["predicted_class"] for it in llama_fs_results])
acc_gz, pw_gz, rw_gz, fw_gz, fm_gz = compute_metrics([it["predicted_class"] for it in gemini_zs_results])
acc_gf, pw_gf, rw_gf, fw_gf, fm_gf = compute_metrics([it["predicted_class"] for it in gemini_fs_results])

tripartite_table = [
    ["Akurasi Keseluruhan", f"{acc_lz*100:.2f}%", f"{acc_lf*100:.2f}%", f"{acc_gz*100:.2f}%", f"{acc_gf*100:.2f}%"],
    ["Weighted Precision", f"{pw_lz*100:.2f}%", f"{pw_lf*100:.2f}%", f"{pw_gz*100:.2f}%", f"{pw_gf*100:.2f}%"],
    ["Weighted Recall", f"{rw_lz*100:.2f}%", f"{rw_lf*100:.2f}%", f"{rw_gz*100:.2f}%", f"{rw_gf*100:.2f}%"],
    ["Weighted F1-Score", f"{fw_lz*100:.2f}%", f"{fw_lf*100:.2f}%", f"{fw_gz*100:.2f}%", f"{fw_gf*100:.2f}%"],
    ["Macro F1-Score", f"{fm_lz*100:.2f}%", f"{fm_lf*100:.2f}%", f"{fm_gz*100:.2f}%", f"{fm_gf*100:.2f}%"],
    ["Latensi Rata-rata", f"{llama_zs_perf['avg_latency_ms']:.1f} ms", f"{llama_fs_perf['avg_latency_ms']:.1f} ms", f"{gemini_zs_perf['avg_latency_ms']:.1f} ms", f"{gemini_fs_perf['avg_latency_ms']:.1f} ms"],
    ["Biaya Komputasi", "Gratis (T4)", "Gratis (T4)", "Free API Tier", "Free API Tier"]
]

print("=" * 95)
print("TABEL KOMPARASI MULTI-LLM: SMALL LLM (LLAMA-3-8B) VS LARGE LLM (GEMINI 1.5) PADA BENCHMARK")
print("=" * 95)
headers = ["Metrik Evaluasi", "Llama-3-8B (Zero)", "Llama-3-8B (Few)", "Gemini 1.5 (Zero)", "Gemini 1.5 (Few)"]
print(tabulate(tripartite_table, headers=headers, tablefmt="github"))

TABEL KOMPARASI MULTI-LLM: SMALL LLM (LLAMA-3-8B) VS LARGE LLM (GEMINI 1.5) PADA BENCHMARK
| Metrik Evaluasi     | Llama-3-8B (Zero)   | Llama-3-8B (Few)   | Gemini 1.5 (Zero)   | Gemini 1.5 (Few)   |
|---------------------|---------------------|--------------------|---------------------|--------------------|
| Akurasi Keseluruhan | 78.48%              | 80.17%             | 5.49%               | 5.49%              |
| Weighted Precision  | 89.96%              | 91.92%             | 0.30%               | 0.30%              |
| Weighted Recall     | 78.48%              | 80.17%             | 5.49%               | 5.49%              |
| Weighted F1-Score   | 80.17%              | 83.14%             | 0.57%               | 0.57%              |
| Macro F1-Score      | 63.69%              | 63.86%             | 0.80%               | 0.80%              |
| Latensi Rata-rata   | 2920.8 ms           | 4489.9 ms          | 103.9 ms            | 93.5 ms            |
| Biaya Komputasi     | Grati

---
### 11. Analisis Kasus Ambiguitas Khusus: Prophet (Nabi) vs Messenger (Rasul)
Membandingkan kepekaan semantik teologis antara Small LLM dan Large LLM.

In [18]:
prophet_messenger_indices = [i for i, y in enumerate(y_true) if y in ["Prophet", "Messenger"]]
y_true_pm = [y_true[i] for i in prophet_messenger_indices]
y_llama_pm = [llama_fs_results[i]["predicted_class"] for i in prophet_messenger_indices]
y_gemini_pm = [gemini_fs_results[i]["predicted_class"] for i in prophet_messenger_indices]

acc_pm_llama = accuracy_score(y_true_pm, y_llama_pm)
acc_pm_gemini = accuracy_score(y_true_pm, y_gemini_pm)

print("\n" + "=" * 75)
print("AKURASI PEMBEDAAN NABI (PROPHET) VS RASUL (MESSENGER):")
print("=" * 75)
print(f"• Jumlah Sampel Uji Kenabian/Kerasulan: {len(prophet_messenger_indices)} entitas")
print(f"• Akurasi Small LLM (Llama-3-8B Few-Shot): {acc_pm_llama*100:.2f}%")
print(f"• Akurasi Large LLM (Gemini 1.5 Few-Shot) : {acc_pm_gemini*100:.2f}%")
print(f"• Selisih Peningkatan (*Scaling Gain*)    : {(acc_pm_gemini - acc_pm_llama)*100:+.2f}%")


AKURASI PEMBEDAAN NABI (PROPHET) VS RASUL (MESSENGER):
• Jumlah Sampel Uji Kenabian/Kerasulan: 40 entitas
• Akurasi Small LLM (Llama-3-8B Few-Shot): 42.50%
• Akurasi Large LLM (Gemini 1.5 Few-Shot) : 0.00%
• Selisih Peningkatan (*Scaling Gain*)    : -42.50%


---
### 12. Pelabelan Entitas Unmapped Korpus 114 Surah (Konsensus Multi-LLM)

In [19]:
print("\n🚀 Melabeli Entitas Unmapped Korpus 114 Surah...")
unmapped_llama, _ = run_llama_inference(unmapped_items, mode="few_shot")
unmapped_gemini, _ = run_gemini_inference(unmapped_items, mode="few_shot")

consensus_rows = []
for l_it, g_it in zip(unmapped_llama, unmapped_gemini):
    agreement = "Cocok (Sepakat)" if l_it["predicted_class"] == g_it["predicted_class"] else "Beda (Discrepancy)"
    consensus_rows.append([
        l_it["surah"],
        l_it["ayah"],
        l_it["entity"],
        l_it["predicted_class"],
        g_it["predicted_class"],
        agreement
    ])

print("\n" + "=" * 75)
print("HASIL KONSENSUS PELABELAN ENTITAS UNMAPPED (SMALL VS LARGE LLM):")
print("=" * 75)
headers_c = ["Surah", "Ayat", "Entitas Teks", "Prediksi Llama-3-8B", "Prediksi Gemini 1.5", "Status Konsensus"]
print(tabulate(consensus_rows, headers=headers_c, tablefmt="github"))

[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🚀 Melabeli Entitas Unmapped Korpus 114 Surah...

🚀 Menjalankan Inferensi Llama-3-8B | Mode: FEW_SHOT | Jumlah: 1 entitas...
  [  1/1] Selesai | Latensi rata-rata: 4375.6 ms

⏱️ Total Waktu: 4.38 detik (4382.2 ms/entitas)
📊 Peak VRAM Penggunaan: 9.57 GB

🚀 Menjalankan Inferensi Large LLM (gemini-1.5-flash) | Mode: FEW_SHOT | Jumlah: 1 entitas...
  [  1/1] Terproses...
⏱️ Total Waktu Gemini: 0.09 detik (93.3 ms/entitas)

HASIL KONSENSUS PELABELAN ENTITAS UNMAPPED (SMALL VS LARGE LLM):
|   Surah |   Ayat | Entitas Teks   | Prediksi Llama-3-8B   | Prediksi Gemini 1.5   | Status Konsensus   |
|---------|--------|----------------|-----------------------|-----------------------|--------------------|
|      33 |     13 | Yasrib         | GeographicalLocation  | Person                | Beda (Discrepancy) |


---
### 13. Ekspor Berkas Komparasi Multi-LLM (`multi_llm_comparative_results.json`)

In [20]:
final_export = {
    "metadata": {
        "experiment": "Multi-LLM Comparative Ablation Study: Small LLM vs Large LLM",
        "benchmark_support": len(benchmark_items),
        "models": {
            "small_llm": "Meta-Llama-3-8B-Instruct (4-bit)",
            "large_llm": GEMINI_MODEL_NAME
        },
        "overall_metrics": {
            "llama_3_8b_zero_shot": {"accuracy": float(acc_lz), "weighted_f1": float(fw_lz), "macro_f1": float(fm_lz)},
            "llama_3_8b_few_shot": {"accuracy": float(acc_lf), "weighted_f1": float(fw_lf), "macro_f1": float(fm_lf)},
            "gemini_1_5_zero_shot": {"accuracy": float(acc_gz), "weighted_f1": float(fw_gz), "macro_f1": float(fm_gz)},
            "gemini_1_5_few_shot": {"accuracy": float(acc_gf), "weighted_f1": float(fw_gf), "macro_f1": float(fm_gf)}
        }
    },
    "benchmark_predictions": {
        "llama_3_8b_few_shot": llama_fs_results,
        "gemini_1_5_few_shot": gemini_fs_results
    },
    "unmapped_consensus": consensus_rows
}

output_filename = "multi_llm_comparative_results.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_export, f, ensure_ascii=False, indent=2)

print(f"\n💾 Berkas '{output_filename}' berhasil dibuat!")
files.download(output_filename)
print("🎉 Berkas komparasi Multi-LLM telah diunduh otomatis ke komputer Anda!")


💾 Berkas 'multi_llm_comparative_results.json' berhasil dibuat!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Berkas komparasi Multi-LLM telah diunduh otomatis ke komputer Anda!
